# Day-2. Section 2.1 — What is tokenization?

- **Tokenization** is the process of converting input text into smaller units called tokens, which are mapped to numerical IDs that the model can process.

## Three concepts you must distinguish
| Concept | Meaning |
| :--- | :--- |
| **Token** | A text unit, which may represent a word, part of a word, punctuation or bytes |
| **Vocabulary** | The set of token IDs recognized by a tokenizer |
| **Token ID** | The numerical identifier assigned to a token |

In [1]:
from transformers import AutoTokenizer

model_id = "HuggingFaceTB/SmolLM2-135M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

text = "I am polishing my AI Engineering"

tokens = tokenizer.tokenize(text)

token_ids = tokenizer.encode(
    text,
    add_special_tokens=False,  # add_special_tokens=False avoids automatically adding model-specific special tokens.
)

print("Tokenizer:", type(tokenizer).__name__)
print("Tokens:", tokens)
print("Token IDs:", token_ids)
print("Vocabulary size:", len(tokenizer))
print("Token count:", len(token_ids))

Tokenizer: GPT2Tokenizer
Tokens: ['I', 'Ġam', 'Ġpolishing', 'Ġmy', 'ĠAI', 'ĠEngineering']
Token IDs: [57, 744, 39493, 957, 5646, 8574]
Vocabulary size: 49152
Token count: 6


## 2.1.1 — Let's verify the actual segmentation

In [2]:
words = [
    "Engineering",
    "Engineeering",
]

for word in words:
    print("Word:", word)
    print("Tokens:", tokenizer.tokenize(word))
    print("Token IDs:", tokenizer.encode(word, add_special_tokens=False))
    print()

Word: Engineering
Tokens: ['Engineering']
Token IDs: [45601]

Word: Engineeering
Tokens: ['Engine', 'eering']
Token IDs: [23353, 12212]



# 2.2 — Byte Pair Encoding (BPE)

**Definition:** Byte Pair Encoding is a tokenization algorithm that repeatedly merges frequently occurring adjacent units to build larger tokens.

In simple terms:
1. Start with small units (characters in our simplified example).
2. Count how frequently adjacent pairs occur.
3. Merge the most frequent pair.
4. Repeat until the desired vocabulary size or merge limit is reached.

In [3]:
from collections import Counter

corpus = ["cat", "cat", "car"]

pairs_count = Counter()

for word in corpus:
    pairs = zip(word, word[1:])
    pairs_count.update(pairs)

    
print(pairs_count)
print("Most frequent:", pairs_count.most_common(1))

Counter({('c', 'a'): 3, ('a', 't'): 2, ('a', 'r'): 1})
Most frequent: [(('c', 'a'), 3)]


In [4]:
tokens = ["c", "a", "t"]
merged = []
i = 0

while i < len(tokens):
    if i + 1 < len(tokens) and tokens[i] == "c" and tokens[i + 1] == "a":
        merged.append("ca")

        i += 2
    else:
        merged.append(tokens[i])
        i += 1

print("Before:", tokens)
print("After:", merged)

Before: ['c', 'a', 't']
After: ['ca', 't']


In [5]:
def merge_pair(tokens, pair):
    merged = []

    i = 0

    while i < len(tokens):
        if i + 1 < len(tokens) and (tokens[i], tokens[i+1]) == pair:
            merged.append(tokens[i] + tokens[i+1])
            i += 2
        else:
            merged.append(tokens[i])
            i += 1
    
    return merged

In [6]:
tokens = ["l", "o", "w", "e", "r"]

result = merge_pair(tokens, ("l", "o"))

print(result)

['lo', 'w', 'e', 'r']


In [7]:
corpus = [
    ["l", "o", "w"],
    ["l", "o", "w"],
    ["l", "o", "w", "e", "r"]
]

for token in corpus:
    result = merge_pair(token, ("l", "o"))

    print(result)

['lo', 'w']
['lo', 'w']
['lo', 'w', 'e', 'r']


In [8]:
corpus = [
['lo', 'w'],
['lo', 'w'],
['lo', 'w', 'e', 'r']
]

pair = ("lo", "w")

updated_corpus = []

for tokens in corpus:
    updated_corpus.append(merge_pair(tokens, pair))

print(updated_corpus)

[['low'], ['low'], ['low', 'e', 'r']]


In [9]:
from collections import Counter

def count_pairs(corpus):
    pair_counts = Counter()

    for tokens in corpus:
        pairs = zip(tokens, tokens[1:])
        pair_counts.update(pairs)

    return pair_counts

In [10]:
pair_counts = count_pairs(updated_corpus)

print(pair_counts)
print("Most frequent:", pair_counts.most_common(1))

Counter({('low', 'e'): 1, ('e', 'r'): 1})
Most frequent: [(('low', 'e'), 1)]


In [11]:
corpus = [
    ["l", "o", "w"],
    ["l", "o", "w"],
    ["l", "o", "w", "e", "r"]
]

num_merges = 2

for step in range(num_merges):

    pair_counts = count_pairs(corpus)
    print(f"STEP:{step+1} | Pair_count: {pair_counts}")
    if not pair_counts:
        break
    

    best_pair = pair_counts.most_common(1)[0][0]

    corpus = [
        merge_pair(tokens, best_pair)
        for tokens in corpus
    ]

    print(f"Merge {step + 1}: {best_pair}")
    print("Updated corpus:", corpus)

STEP:1 | Pair_count: Counter({('l', 'o'): 3, ('o', 'w'): 3, ('w', 'e'): 1, ('e', 'r'): 1})
Merge 1: ('l', 'o')
Updated corpus: [['lo', 'w'], ['lo', 'w'], ['lo', 'w', 'e', 'r']]
STEP:2 | Pair_count: Counter({('lo', 'w'): 3, ('w', 'e'): 1, ('e', 'r'): 1})
Merge 2: ('lo', 'w')
Updated corpus: [['low'], ['low'], ['low', 'e', 'r']]


In [12]:
corpus_words = ["low", "low", "lower"]
merge_rules = [("l", "o"), ("lo", "w")]
vocab = set("".join(corpus_words))

for left, right in merge_rules:
    vocab.add(left + right)

vocab = sorted(vocab)

print("Vocabulary:", vocab)
print("Vocabulary size:", len(vocab))

Vocabulary: ['e', 'l', 'lo', 'low', 'o', 'r', 'w']
Vocabulary size: 7


# 3. Byte Pair Encoding, step by step


**Training pipeline**

`CORPUS` → `pre_tokenize` → `get_word_freqs` → `build_initial_splits` → `build_base_vocab` → loop [`count_pairs` → `find_best_pair` → `merge_pair`] → ordered `merges`

**Inference pipeline**

`text` → `pre_tokenize` → `tokenize_word` (apply merges by rank) → `encode` (ids) → `decode` (back to text)

This is classic word-level BPE (Sennrich et al., 2016) with an end-of-word marker. GPT-2-style byte-level BPE is the same algorithm run over bytes instead of characters, with a fancier pre-tokenizer.

In [13]:
import re
from collections import Counter

# End-of-word marker. Without it, 'est' at the END of 'lowest' and 'est' INSIDE
# 'estimate' would be the same symbol, so BPE could not learn suffixes like 'est</w>'.
# It also lets decode() know where to put spaces back.
END = '</w>'

## 3.0 Corpus

It's kept small on purpose. The words share stems (`low`, `new`, `wide`) and suffixes (`-er`, `-est`), so you can check whether BPE finds them. It mostly won't, and section 9 explains why. Add your own lines and re-run the training cell (section 9) to see how the merges change.

In [14]:
CORPUS = [
    'the lowest price is lower than the low price',
    'the newest model is newer than the new model',
    'the widest road is wider than the wide road',
    'lower and lowest, newer and newest, wider and widest!',
    'tokenization helps models handle new words',
]
print(f'{len(CORPUS)} lines')

5 lines


## 3.1 `pre_tokenize(text)`

**What it does:** lowercases the text and splits it into words and punctuation marks.

**Why it exists:** BPE never merges across these boundaries. The output of this step sets the outer limit on every token, so no token can ever span `lower` and `,`.

**Input → output:** `str` → `list[str]`

**Try breaking it:** remove `.lower()` and see how `Lower` and `lower` become separate words with separate counts. Or change the regex so punctuation stays attached to words.

In [15]:
def pre_tokenize(text, verbose=True):
    '''Lowercase, then split into runs of word characters or single punctuation marks.'''
    words = re.findall(r'\w+|[^\w\s]', text.lower())   # \w+ = a word, [^\w\s] = one punctuation char
    if verbose:
        print(f'[pre_tokenize] {text!r}')
        print(f'               -> {words}')
    return words

In [16]:
# TEST: punctuation becomes its own word, case is folded
pre_tokenize('Lower and LOWEST, newer!')

[pre_tokenize] 'Lower and LOWEST, newer!'
               -> ['lower', 'and', 'lowest', ',', 'newer', '!']


['lower', 'and', 'lowest', ',', 'newer', '!']

## 3.2 `get_word_freqs(corpus)`

**What it does:** counts how often each unique word appears across the whole corpus.

**Why it exists:** BPE works on unique words weighted by their frequency, not on running text. A word that appears 1,000 times is processed once and its pair counts are multiplied by 1,000. This is the main reason BPE training is fast.

**Input → output:** `list[str]` → `Counter{word: count}`

In [17]:
def get_word_freqs(corpus, verbose=True):
    '''Pre-tokenize every line and count unique words.'''
    freqs = Counter()
    for line in corpus:
        freqs.update(pre_tokenize(line, verbose=False))
    if verbose:
        print(f'[get_word_freqs] {len(freqs)} unique words, {sum(freqs.values())} total')
        for word, count in freqs.most_common():
            print(f'    {word!r:15} x{count}')
    return freqs

In [18]:
# TEST: on the real corpus
word_freqs = get_word_freqs(CORPUS)

[get_word_freqs] 23 unique words, 45 total
    'the'           x6
    'is'            x3
    'than'          x3
    'and'           x3
    'lowest'        x2
    'price'         x2
    'lower'         x2
    'newest'        x2
    'model'         x2
    'newer'         x2
    'new'           x2
    'widest'        x2
    'road'          x2
    'wider'         x2
    ','             x2
    'low'           x1
    'wide'          x1
    '!'             x1
    'tokenization'  x1
    'helps'         x1
    'models'        x1
    'handle'        x1
    'words'         x1


## 3.3 `build_initial_splits(word_freqs)`

**What it does:** breaks every word into single characters and appends `</w>` at the end.

**Why it exists:** this is the starting state for training. Every merge from here on fuses two adjacent symbols inside one of these lists.

**Input → output:** `{word: count}` → `{word: [symbols]}`

In [19]:
def build_initial_splits(word_freqs, verbose=True):
    '''word -> list of characters + end-of-word marker.'''
    splits = {word: list(word) + [END] for word in word_freqs}
    if verbose:
        print(f'[build_initial_splits] {len(splits)} words split into characters')
        for word in list(splits)[:5]:
            print(f'    {word!r:15} {splits[word]}')
    return splits

In [20]:
# TEST: a tiny standalone toy, independent of CORPUS. Later test cells reuse it.
toy_freqs = {'low': 5, 'lower': 2, 'lowest': 3}
toy_splits = build_initial_splits(toy_freqs)

[build_initial_splits] 3 words split into characters
    'low'           ['l', 'o', 'w', '</w>']
    'lower'         ['l', 'o', 'w', 'e', 'r', '</w>']
    'lowest'        ['l', 'o', 'w', 'e', 's', 't', '</w>']


## 3.4 `build_base_vocab(splits)`

**What it does:** collects every distinct starting symbol, which is every character plus `</w>`.

**Why it exists:** this is the floor of the vocabulary. Any word made only of these characters can always be tokenized, even if no merge applies. A character outside this set becomes `<unk>` at encode time (section 12).

**Input → output:** `{word: [symbols]}` → `list[str]` (sorted, so runs are reproducible)

In [21]:
def build_base_vocab(splits, verbose=True):
    '''All distinct symbols in the initial splits.'''
    vocab = sorted({symbol for symbols in splits.values() for symbol in symbols})
    if verbose:
        print(f'[build_base_vocab] {len(vocab)} base symbols: {vocab}')
    return vocab

In [22]:
# TEST
build_base_vocab(toy_splits)

[build_base_vocab] 8 base symbols: ['</w>', 'e', 'l', 'o', 'r', 's', 't', 'w']


['</w>', 'e', 'l', 'o', 'r', 's', 't', 'w']

## 3.5 `merge_symbols(symbols, pair)` (helper)

**What it does:** inside one list of symbols, replaces every adjacent `(a, b)` with the single symbol `a+b`.

**Why it exists:** training (`merge_pair`) and inference (`tokenize_word`) both need exactly this operation. Keeping it in one place guarantees the two paths agree.

**Edge case:** it scans left to right and skips past each merge, so `a a a` merged on `(a, a)` gives `aa a`, not `aa aa`.

In [23]:
def merge_symbols(symbols, pair):
    '''Replace every adjacent occurrence of pair (a, b) with a+b in ONE symbol list.'''
    a, b = pair
    out, i = [], 0
    while i < len(symbols):
        if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
            out.append(a + b)   # fuse the pair into one symbol
            i += 2              # skip both halves
        else:
            out.append(symbols[i])
            i += 1
    return out

In [24]:
# TEST
print(merge_symbols(['l', 'o', 'w', 'e', 's', 't', END], ('e', 's')))
print(merge_symbols(['a', 'a', 'a'], ('a', 'a')))   # overlap case -> ['aa', 'a']

['l', 'o', 'w', 'es', 't', '</w>']
['aa', 'a']


## 3.6 `count_pairs(splits, word_freqs)`

**What it does:** counts every adjacent symbol pair across all words, weighted by word frequency.

**Why it exists:** these counts are the evidence BPE uses to decide what to merge. A pair inside a word that appears 5 times is counted 5 times.

**Input → output:** `{word: [symbols]}`, `{word: count}` → `Counter{(a, b): count}`

**Check by hand:** in the toy, `('l','o')` appears in all three words, so its count is 5 + 2 + 3 = 10.

In [25]:
def count_pairs(splits, word_freqs, verbose=True, top_k=8):
    '''Frequency-weighted count of adjacent symbol pairs.'''
    pair_counts = Counter()
    for word, symbols in splits.items():
        freq = word_freqs[word]
        for pair in zip(symbols, symbols[1:]):   # (s0,s1), (s1,s2), ...
            pair_counts[pair] += freq
    if verbose:
        print(f'[count_pairs] {len(pair_counts)} distinct pairs; top {top_k}:')
        for pair, count in pair_counts.most_common(top_k):
            print(f'    {pair!s:24} {count}')
    return pair_counts

In [26]:

# TEST

toy_pairs = count_pairs(toy_splits, toy_freqs)

[count_pairs] 9 distinct pairs; top 8:
    ('l', 'o')               10
    ('o', 'w')               10
    ('w', '</w>')            5
    ('w', 'e')               5
    ('e', 's')               3
    ('s', 't')               3
    ('t', '</w>')            3
    ('e', 'r')               2


## 3.7 `find_best_pair(pair_counts)`

**What it does:** returns the most frequent pair. Ties are broken alphabetically, so every run gives the same result.

**Why it exists:** this one line is the entire BPE selection rule. Replace it with `count(ab) / (count(a) * count(b))` and you have essentially turned this into WordPiece training.

**Input → output:** `Counter{pair: count}` → `(a, b)`, or `None` if there are no pairs left

In [27]:
def find_best_pair(pair_counts, verbose=True):
    '''Highest count wins; ties broken alphabetically for reproducibility.'''
    if not pair_counts:
        if verbose:
            print('[find_best_pair] no pairs left')
        return None
    best = min(pair_counts, key=lambda p: (-pair_counts[p], p))
    if verbose:
        print(f'[find_best_pair] {best} with count {pair_counts[best]}')
    return best

In [28]:
# TEST
find_best_pair(toy_pairs)
find_best_pair(Counter({('c', 'd'): 3, ('a', 'b'): 3}))   # tie -> ('a', 'b')

[find_best_pair] ('l', 'o') with count 10
[find_best_pair] ('a', 'b') with count 3


('a', 'b')

## 3.8 `merge_pair(pair, splits)`

**What it does:** applies one merge to every word in the corpus.

**Why it exists:** after this runs, the next call to `count_pairs` sees the new symbol, so new pairs become possible, such as `('lo', 'w')`. This is how BPE builds longer tokens out of shorter ones.

**Design choice:** it returns a **new** dict instead of mutating the old one, so you can compare before and after in a test cell.

In [29]:
def merge_pair(pair, splits, verbose=True):
    '''Apply one merge to every word; returns a NEW splits dict.'''
    new_splits = {word: merge_symbols(symbols, pair) for word, symbols in splits.items()}
    if verbose:
        changed = [w for w in splits if new_splits[w] != splits[w]]
        print(f'[merge_pair] {pair[0]!r} + {pair[1]!r} -> {pair[0] + pair[1]!r}; changed {len(changed)} word(s)')
        for w in changed[:6]:
            print(f'    {w!r:15} {splits[w]} -> {new_splits[w]}')
    return new_splits

In [30]:
# TEST: do two merges by hand, the way the training loop will
step1 = merge_pair(('l', 'o'), toy_splits)
count_pairs(step1, toy_freqs, top_k=3)
step2 = merge_pair(('lo', 'w'), step1)

[merge_pair] 'l' + 'o' -> 'lo'; changed 3 word(s)
    'low'           ['l', 'o', 'w', '</w>'] -> ['lo', 'w', '</w>']
    'lower'         ['l', 'o', 'w', 'e', 'r', '</w>'] -> ['lo', 'w', 'e', 'r', '</w>']
    'lowest'        ['l', 'o', 'w', 'e', 's', 't', '</w>'] -> ['lo', 'w', 'e', 's', 't', '</w>']
[count_pairs] 8 distinct pairs; top 3:
    ('lo', 'w')              10
    ('w', '</w>')            5
    ('w', 'e')               5
[merge_pair] 'lo' + 'w' -> 'low'; changed 3 word(s)
    'low'           ['lo', 'w', '</w>'] -> ['low', '</w>']
    'lower'         ['lo', 'w', 'e', 'r', '</w>'] -> ['low', 'e', 'r', '</w>']
    'lowest'        ['lo', 'w', 'e', 's', 't', '</w>'] -> ['low', 'e', 's', 't', '</w>']


## 3.9 `train_bpe(corpus, num_merges)`

**What it does:** runs the loop count → pick best → merge → record, `num_merges` times.

**Returns:**
- `merges`: the ordered list of pairs. **The order matters**, because it is the priority used at inference time.
- `vocab`: the base characters plus one new symbol per merge, so `len(vocab) = base + len(merges)`.
- `splits`: how each training word ended up segmented.

**Stopping rule:** training stops early when the best pair occurs fewer than `min_frequency` times. Merging a pair seen only once just memorizes one word. Production tokenizers expose the same knob, for example `min_frequency` in Hugging Face `tokenizers`.

**Try:** `num_merges=5` versus `50`, or `min_frequency=1`. Watch whole words such as `tokenization</w>` start to appear.

In [31]:
def train_bpe(corpus, num_merges=30, min_frequency=2, verbose=True):
    '''Learn an ordered list of merges from a corpus.'''
    word_freqs = get_word_freqs(corpus, verbose=False)
    splits = build_initial_splits(word_freqs, verbose=False)
    vocab = build_base_vocab(splits, verbose=False)
    merges = []
    if verbose:
        print(f'[train_bpe] {len(word_freqs)} unique words, base vocab = {len(vocab)} symbols')

    for step in range(1, num_merges + 1):
        pair_counts = count_pairs(splits, word_freqs, verbose=False)
        best = find_best_pair(pair_counts, verbose=False)
        if best is None or pair_counts[best] < min_frequency:
            if verbose:
                print(f'[train_bpe] stop at step {step}: best pair count < {min_frequency}')
            break
        splits = merge_pair(best, splits, verbose=False)
        merges.append(best)                 # record the rule, in order
        vocab.append(best[0] + best[1])     # the new symbol joins the vocab
        if verbose:
            new = best[0] + best[1]
            print(f'  merge {step:2}: {best[0]!r:9} + {best[1]!r:9} -> {new!r:13} count={pair_counts[best]}')

    if verbose:
        print(f'[train_bpe] learned {len(merges)} merges; final vocab = {len(vocab)} symbols')
    return merges, vocab, splits

In [32]:
# TEST: train on the real corpus
merges, vocab, final_splits = train_bpe(CORPUS, num_merges=30)

[train_bpe] 23 unique words, base vocab = 20 symbols
  merge  1: 'e'       + '</w>'    -> 'e</w>'       count=10
  merge  2: 't'       + 'h'       -> 'th'          count=9
  merge  3: 'w'       + 'e'       -> 'we'          count=8
  merge  4: 'a'       + 'n'       -> 'an'          count=7
  merge  5: 'd'       + 'e'       -> 'de'          count=7
  merge  6: 'n'       + 'e'       -> 'ne'          count=6
  merge  7: 'r'       + '</w>'    -> 'r</w>'       count=6
  merge  8: 's'       + '</w>'    -> 's</w>'       count=6
  merge  9: 's'       + 't'       -> 'st'          count=6
  merge 10: 'st'      + '</w>'    -> 'st</w>'      count=6
  merge 11: 'th'      + 'e</w>'   -> 'the</w>'     count=6
  merge 12: 'd'       + '</w>'    -> 'd</w>'       count=5
  merge 13: 'l'       + 'o'       -> 'lo'          count=5
  merge 14: 'w'       + 'i'       -> 'wi'          count=5
  merge 15: 'lo'      + 'we'      -> 'lowe'        count=4
  merge 16: 'ne'      + 'we'      -> 'newe'        count=4
  

In [33]:
for w in ['lowest', 'lower', 'newest', 'widest', 'models', 'tokenization']:
    print(f'{w!r:15} {final_splits[w]}')

'lowest'        ['lowe', 'st</w>']
'lower'         ['lower</w>']
'newest'        ['newe', 'st</w>']
'widest'        ['wide', 'st</w>']
'models'        ['model', 's</w>']
'tokenization'  ['t', 'o', 'k', 'e', 'n', 'i', 'z', 'a', 't', 'i', 'o', 'n', '</w>']


**What to notice in the output above:**

- The split is `lowe + st</w>`, not `low + est</w>`. BPE optimizes for frequency, not linguistics. `we` is frequent (it appears in lower, newer, newest and more), so it got merged early and the stem boundary was lost. Unigram LM tokenization tends to produce more morpheme-like splits.
- `tokenization` stays as single characters because it appears only once, and `min_frequency=2` blocks every pair that is unique to it.
- `lower` became a single token (`lower</w>`): with only 23 unique words, a few dozen merges are enough to start memorizing whole words.

## 3.10. `tokenize_word(word, merges)`

**What it does:** segments **one new word** using the learned merges.

**How:** it starts from characters plus `</w>`. On each pass it finds every adjacent pair that is in `merges`, applies the one learned **earliest** (lowest rank), and repeats until no learned pair remains. This reproduces the order in which merges happened during training, and it is the same loop as GPT-2's `bpe()` function.

**Contrast with WordPiece:** WordPiece throws the merge list away and does a greedy longest match against the vocabulary. BPE replays its merge rules.

In [34]:
def tokenize_word(word, merges, verbose=True):
    '''Apply learned merges to one word, earliest-learned first.'''
    ranks = {pair: i for i, pair in enumerate(merges)}   # pair -> priority (lower = learned earlier)
    symbols = list(word) + [END]
    if verbose:
        print(f'[tokenize_word] {word!r}: start {symbols}')
    while len(symbols) > 1:
        candidates = [(ranks[p], p) for p in zip(symbols, symbols[1:]) if p in ranks]
        if not candidates:
            break                       # no learned merge applies any more
        rank, pair = min(candidates)    # earliest-learned merge wins
        symbols = merge_symbols(symbols, pair)
        if verbose:
            print(f'    rank {rank:2} {pair!s:24} -> {symbols}')
    return symbols

In [35]:
# TEST: a word seen in training
tokenize_word('lowest', merges)

[tokenize_word] 'lowest': start ['l', 'o', 'w', 'e', 's', 't', '</w>']
    rank  2 ('w', 'e')               -> ['l', 'o', 'we', 's', 't', '</w>']
    rank  8 ('s', 't')               -> ['l', 'o', 'we', 'st', '</w>']
    rank  9 ('st', '</w>')           -> ['l', 'o', 'we', 'st</w>']
    rank 12 ('l', 'o')               -> ['lo', 'we', 'st</w>']
    rank 14 ('lo', 'we')             -> ['lowe', 'st</w>']


['lowe', 'st</w>']

In [36]:
# TEST: an unseen word still breaks into pieces BPE has learned
tokenize_word('slowest', merges)

[tokenize_word] 'slowest': start ['s', 'l', 'o', 'w', 'e', 's', 't', '</w>']
    rank  2 ('w', 'e')               -> ['s', 'l', 'o', 'we', 's', 't', '</w>']
    rank  8 ('s', 't')               -> ['s', 'l', 'o', 'we', 'st', '</w>']
    rank  9 ('st', '</w>')           -> ['s', 'l', 'o', 'we', 'st</w>']
    rank 12 ('l', 'o')               -> ['s', 'lo', 'we', 'st</w>']
    rank 14 ('lo', 'we')             -> ['s', 'lowe', 'st</w>']


['s', 'lowe', 'st</w>']

## 3.11 `tokenize(text, merges)`

**What it does:** pre-tokenizes the text, then runs `tokenize_word` on each word and concatenates the results.

**Input → output:** `str` → `list[str]` (tokens)

In [37]:
def tokenize(text, merges, verbose=True):
    '''Full text -> list of BPE tokens.'''
    words = pre_tokenize(text, verbose=verbose)
    tokens = []
    for w in words:
        tokens.extend(tokenize_word(w, merges, verbose=False))
    if verbose:
        print(f'[tokenize] {len(words)} words -> {len(tokens)} tokens')
        print(f'    {tokens}')
    return tokens

In [38]:
# TEST
tokens = tokenize('The newest, widest model is lower!', merges)

[pre_tokenize] 'The newest, widest model is lower!'
               -> ['the', 'newest', ',', 'widest', 'model', 'is', 'lower', '!']
[tokenize] 8 words -> 12 tokens
    ['the</w>', 'newe', 'st</w>', ',</w>', 'wide', 'st</w>', 'model', '</w>', 'is</w>', 'lower</w>', '!', '</w>']


## 12. `build_token_to_id`, `encode`, `decode`

**`build_token_to_id(vocab)`** gives every token an integer id. Id 0 is reserved for `<unk>`.

**`encode(text)`** turns text into tokens and then into ids. A character never seen in training, such as `q` here, has no entry and maps to `<unk>`.

**`decode(ids)`** turns ids back into tokens, joins them, and replaces each `</w>` with a space.

**Why decode is lossy here:** casing and the original spacing around punctuation are lost, because `pre_tokenize` threw them away. That limitation, plus `<unk>`, is exactly why GPT-2 moved to **byte-level** BPE. With 256 byte symbols as the base vocabulary, nothing is ever unknown and decoding is exact.

In [39]:
def build_token_to_id(vocab, verbose=True):
    '''Assign integer ids; id 0 is reserved for <unk>.'''
    id_to_token = ['<unk>'] + list(vocab)
    token_to_id = {tok: i for i, tok in enumerate(id_to_token)}
    if verbose:
        print(f'[build_token_to_id] {len(id_to_token)} ids; last 8: {id_to_token[-8:]}')
    return token_to_id, id_to_token


def encode(text, merges, token_to_id, verbose=True):
    '''text -> token ids (unknown tokens -> <unk> id).'''
    tokens = tokenize(text, merges, verbose=False)
    unk_id = token_to_id['<unk>']
    ids = [token_to_id.get(tok, unk_id) for tok in tokens]
    if verbose:
        print(f'[encode] {text!r}')
        for tok, i in zip(tokens, ids):
            print(f'    {tok!r:14} -> {i}' + ('   <-- unknown' if i == unk_id else ''))
    return ids


def decode(ids, id_to_token, verbose=True):
    '''token ids -> text. Each </w> becomes a space.'''
    tokens = [id_to_token[i] for i in ids]
    text = ''.join(tokens).replace(END, ' ').strip()
    if verbose:
        print(f'[decode] {tokens}')
        print(f'      -> {text!r}')
    return text

In [40]:
# TEST: round trip
token_to_id, id_to_token = build_token_to_id(vocab)
ids = encode('The lowest, newest model', merges, token_to_id)
decode(ids, id_to_token)

[build_token_to_id] 51 ids; last 8: ['model', 'than</w>', 'w</w>', ',</w>', 'ad</w>', 'ce</w>', 'ice</w>', 'lower</w>']
[encode] 'The lowest, newest model'
    'the</w>'      -> 31
    'lowe'         -> 35
    'st</w>'       -> 30
    ',</w>'        -> 46
    'newe'         -> 36
    'st</w>'       -> 30
    'model'        -> 43
    '</w>'         -> 3
[decode] ['the</w>', 'lowe', 'st</w>', ',</w>', 'newe', 'st</w>', 'model', '</w>']
      -> 'the lowest , newest model'


'the lowest , newest model'

In [41]:
# TEST: unknown character ('q' never appears in CORPUS)
ids = encode('quick lower', merges, token_to_id)
decode(ids, id_to_token)

[encode] 'quick lower'
    'q'            -> 0   <-- unknown
    'u'            -> 0   <-- unknown
    'i'            -> 9
    'c'            -> 5
    'k'            -> 10
    '</w>'         -> 3
    'lower</w>'    -> 50
[decode] ['<unk>', '<unk>', 'i', 'c', 'k', '</w>', 'lower</w>']
      -> '<unk><unk>ick lower'


'<unk><unk>ick lower'

# 2.4.1 — Practical: Longest-Match-First(WordPiece)

In [42]:
vocab = {
    "un",
    "under",
    "##break",
    "##able",
    "##breakable",
    }

word = "unbreakable"

matches = [
    token for token in vocab
    if not token.startswith("##")
    and word.startswith(token)
]

longest_match = max(matches, key=len)

print("Longest initial match:", longest_match)

Longest initial match: un


# 2.4.2 — Implementing WordPiece continuation matching

In [43]:
word = "unbreakable"
first_token = "unbreak"

remaining = word[len(first_token):]

print("First token:", first_token)
print("Remaining text:", remaining)

First token: unbreak
Remaining text: able


**step-2**

In [44]:
matches = [
    token for token in vocab
    if token.startswith("##")
    and remaining.startswith(token[2:])
]
print(matches)
longest_match = max(matches, key=len)

print("Continuation token:", longest_match)

['##able']
Continuation token: ##able


# 2.4.3 — Build a reusable WordPiece tokenizer

In [45]:
def wordpiece_tokenize(word, vocab):
    result = []
    remaining = word


    while remaining:
        prefix = "" if not result else "##"

        matches = [
            token for token in vocab
            if token.startswith(prefix)
            and (prefix != "" or not token.startswith("##"))
            and remaining.startswith(token[len(prefix):])
        ]

        if not matches:
            return ["[UNK]"]

        best_match = max(matches, key=len)
        result.append(best_match)

        remaining = remaining[len(best_match) - len(prefix):]
    
    return result

In [46]:
vocab = {
    "un",
    "unbreak",
    # "##break",
    # "##able",
    "##breakable",
}

print(wordpiece_tokenize("unbreakable", vocab))

['[UNK]']


**Calculate WordPiece Scores**

In [47]:
pairs = {
    ("a", "b"):(10, 20, 20),
    ("x", "y"):(5, 20, 5)
}

for pair, (pair_freq, freq_a, freq_b) in pairs.items():
    score = pair_freq/(freq_a*freq_b)

    print(f"Pair: {pair}, Score: {score:.3f}")

Pair: ('a', 'b'), Score: 0.025
Pair: ('x', 'y'), Score: 0.050


# 2.5.0 — Practical: Unigram Segmentation

In [51]:
from math import prod

token_probs = {
    "play": 0.05,
    "ing": 0.4,
    "pla": 0.5,
    "y": 0.2,
}

segmentations = [
    ["play", "ing"],
    ["pla", "y", "ing"],
]

for tokens in segmentations:
    score = prod(token_probs[token] for token in tokens)

    print(f"Tokens: {tokens}, Score: {score:.4f}")

Tokens: ['play', 'ing'], Score: 0.0200
Tokens: ['pla', 'y', 'ing'], Score: 0.0400


# 2.5.1 — Practical: Automatically select the best segmentation

In [54]:
from math import prod

token_probs = {
    "play": 0.05,
    "ing": 0.4,
    "pla": 0.5,
    "y": 0.2,
}

segmentations = [
    ["play", "ing"],
    ["pla", "y", "ing"],
]

def segmentation_score(tokens):
    return prod(token_probs[token] for token in tokens)

best_segmentation = max(segmentations, key=segmentation_score)

print("Best segmentation:", best_segmentation)
print("Score:", segmentation_score(best_segmentation))

Best segmentation: ['pla', 'y', 'ing']
Score: 0.04000000000000001


# 2.5.2 — Practical: Simulate Vocabulary Pruning

In [59]:
from math import prod

token_probs = {
    "play": 0.05,
    "pla": 0.5,
    "y": 0.2,
    "ing": 0.4
}

segmentations = [
    ["play", "ing"],
    ["pla", "y", "ing"]
]


def best_score(vocab):
    valid = [
        tokens for tokens in segmentations
        if all(token in vocab for token in tokens)
    ]

    if not valid:
        return 0.0

    return max(
        prod(token_probs[token] for token in tokens)
        for tokens in valid
    )


vocab = set(token_probs)
print(vocab)
before = best_score(vocab=vocab)
vocab.remove("pla")
print(vocab)
after = best_score(vocab=vocab)

print("Before pruning:", before)
print("After pruning:", after)

{'play', 'ing', 'pla', 'y'}
{'play', 'ing', 'y'}
Before pruning: 0.04000000000000001
After pruning: 0.020000000000000004


# 2.6: Comparing Real Tokenizers

In [60]:
from transformers import AutoTokenizer

smollm_tokenizer = AutoTokenizer.from_pretrained(
    "HuggingFaceTB/SmolLM2-135M-Instruct"
)

bert_tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

c:\Users\rudra\GENAI_2026\ai-engineering-lab\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rudra\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

**Step 2 — Compare the same text**

In [63]:
text = "I am learning tokenization"

for name, tok in [
    ("SmolLM2", smollm_tokenizer),
    ("BERT", bert_tokenizer),
]:
    tokens = tok.tokenize(text)

    print(name)
    print("Tokens:", tokens)
    print("Token count:", len(tokens))
    print()

SmolLM2
Tokens: ['I', 'Ġam', 'Ġlearning', 'Ġtoken', 'ization']
Token count: 5

BERT
Tokens: ['i', 'am', 'learning', 'token', '##ization']
Token count: 5



**Compare Python code**

In [64]:
text = "def calculate_total(a, b): return a + b"

for name, tok in [
    ("SmolLM2", smollm_tokenizer),
    ("BERT", bert_tokenizer),
]:
    tokens = tok.tokenize(text)

    print(f"\n{name}")
    print("Tokens:", tokens)
    print("Token count:", len(tokens))


SmolLM2
Tokens: ['def', 'Ġcalculate', '_', 'total', '(', 'a', ',', 'Ġb', '):', 'Ġreturn', 'Ġa', 'Ġ+', 'Ġb']
Token count: 13

BERT
Tokens: ['def', 'calculate', '_', 'total', '(', 'a', ',', 'b', ')', ':', 'return', 'a', '+', 'b']
Token count: 14


In [65]:
text = "):"

print("SmolLM2:", smollm_tokenizer.tokenize(text))
print("BERT:", bert_tokenizer.tokenize(text))

print(
    "SmolLM2 token ID:",
    smollm_tokenizer.convert_tokens_to_ids("):")
)

SmolLM2: ['):']
BERT: [')', ':']
SmolLM2 token ID: 727


**Number Tokenization**

In [66]:
numbers = [
    "123",
    "123456789",
    "3.14159",
    "2026-10-17",
]

for number in numbers:
    print(f"\nInput: {number}")

    for name, tok in [
        ("SmolLM2", smollm_tokenizer),
        ("BERT", bert_tokenizer),
    ]:
        tokens = tok.tokenize(number)

        print(f"{name}: {tokens}")
        print(f"Token count: {len(tokens)}")


Input: 123
SmolLM2: ['1', '2', '3']
Token count: 3
BERT: ['123']
Token count: 1

Input: 123456789
SmolLM2: ['1', '2', '3', '4', '5', '6', '7', '8', '9']
Token count: 9
BERT: ['123', '##45', '##6', '##7', '##8', '##9']
Token count: 6

Input: 3.14159
SmolLM2: ['3', '.', '1', '4', '1', '5', '9']
Token count: 7
BERT: ['3', '.', '141', '##59']
Token count: 4

Input: 2026-10-17
SmolLM2: ['2', '0', '2', '6', '-', '1', '0', '-', '1', '7']
Token count: 10
BERT: ['202', '##6', '-', '10', '-', '17']
Token count: 6


**Indian-language tokenization**

In [67]:
text = "नमस्ते, मैं हिंदी सीख रहा हूँ।"

for name, tok in [
    ("SmolLM2", smollm_tokenizer),
    ("BERT", bert_tokenizer),
]:
    token_ids = tok.encode(
        text,
        add_special_tokens=False
    )

    tokens = tok.convert_ids_to_tokens(token_ids)
    decoded = tok.decode(
        token_ids,
        skip_special_tokens=True
    )

    print(f"\n{name}")
    print("Tokens:", tokens)
    print("Token count:", len(token_ids))
    print("Decoded:", decoded)
    print("UNK count:", token_ids.count(tok.unk_token_id)
          if tok.unk_token_id is not None else 0)


SmolLM2
Tokens: ['à¤¨', 'à¤®', 'à¤¸', 'à¥į', 'à¤¤', 'à¥ĩ', ',', 'Ġà¤', '®', 'à¥', 'Ī', 'à¤Ĥ', 'Ġà¤', '¹', 'à¤¿', 'à¤Ĥ', 'à¤¦', 'à¥Ģ', 'Ġà¤¸', 'à¥Ģ', 'à¤', 'ĸ', 'Ġà¤', '°', 'à¤', '¹', 'à¤¾', 'Ġà¤', '¹', 'à¥', 'Ĥ', 'à¤', 'ģ', 'à¥', '¤']
Token count: 35
Decoded: नमस्ते, मैं हिंदी सीख रहा हूँ।
UNK count: 0

BERT
Tokens: ['न', '##म', '##स', '##त', ',', 'म', 'ह', '##ि', '##द', '##ी', 'स', '##ी', '##ख', 'र', '##ह', '##ा', 'ह', '।']
Token count: 18
Decoded: नमसत, म हिदी सीख रहा ह ।
UNK count: 0


In [68]:
text = "नमस्ते, मैं हिंदी सीख रहा हूँ।"

for name, tok in [
    ("SmolLM2", smollm_tokenizer),
    ("BERT", bert_tokenizer),
]:
    ids = tok.encode(text, add_special_tokens=False)
    decoded = tok.decode(ids, skip_special_tokens=True)

    print(name)
    print("Exact round-trip:", text == decoded)
    print("Decoded:", decoded)

SmolLM2
Exact round-trip: True
Decoded: नमस्ते, मैं हिंदी सीख रहा हूँ।
BERT
Exact round-trip: False
Decoded: नमसत, म हिदी सीख रहा ह ।


**Understanding Special Tokens**

In [69]:
for name, tok in [
    ("SmolLM2", smollm_tokenizer),
    ("BERT", bert_tokenizer),
]:
    print(f"\nTokenizer: {name}")

    print("BOS:", tok.bos_token, tok.bos_token_id)
    print("EOS:", tok.eos_token, tok.eos_token_id)
    print("PAD:", tok.pad_token, tok.pad_token_id)
    print("UNK:", tok.unk_token, tok.unk_token_id)

    print("Special tokens map:", tok.special_tokens_map)


Tokenizer: SmolLM2
BOS: <|im_start|> 1
EOS: <|im_end|> 2
PAD: <|im_end|> 2
UNK: <|endoftext|> 0
Special tokens map: {'bos_token': '<|im_start|>', 'eos_token': '<|im_end|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|im_end|>'}

Tokenizer: BERT
BOS: None None
EOS: None None
PAD: [PAD] 0
UNK: [UNK] 100
Special tokens map: {'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}


**How are special tokens added?**

In [72]:
text = "I love AI"

for name, tok in [
    ("SmolLM2", smollm_tokenizer),
    ("BERT", bert_tokenizer),
]:
    without_special = tok.encode(
        text,
        add_special_tokens=False
    )

    with_special = tok.encode(
        text,
        add_special_tokens=True
    )

    print(f"\n{name}")
    print("Without:", tok.convert_ids_to_tokens(without_special))
    print("With:", tok.convert_ids_to_tokens(with_special))
    print("Difference:", len(with_special) - len(without_special))


SmolLM2
Without: ['I', 'Ġlove', 'ĠAI']
With: ['I', 'Ġlove', 'ĠAI']
Difference: 0

BERT
Without: ['i', 'love', 'ai']
With: ['[CLS]', 'i', 'love', 'ai', '[SEP]']
Difference: 2


In [73]:
text = "I love AI"

manual_text = f"[CLS] {text} [SEP]"

tokens = bert_tokenizer.encode(
    manual_text,
    add_special_tokens=True
)

print("Tokens:", bert_tokenizer.convert_ids_to_tokens(tokens))
print("Token count:", len(tokens))

Tokens: ['[CLS]', '[CLS]', 'i', 'love', 'ai', '[SEP]', '[SEP]']
Token count: 7


**Batch padding using BERT**

In [74]:
texts = [
    "I love AI",
    "I love learning about AI"
]

batch = bert_tokenizer(
    texts,
    padding=True,
    return_tensors="pt"
)

print("Input IDs shape:", batch["input_ids"].shape)

for i in range(len(texts)):
    print(f"\nInput {i + 1}")
    print("Tokens:", bert_tokenizer.convert_ids_to_tokens(
        batch["input_ids"][i]
    ))
    print("Attention mask:", batch["attention_mask"][i].tolist())

Input IDs shape: torch.Size([2, 7])

Input 1
Tokens: ['[CLS]', 'i', 'love', 'ai', '[SEP]', '[PAD]', '[PAD]']
Attention mask: [1, 1, 1, 1, 1, 0, 0]

Input 2
Tokens: ['[CLS]', 'i', 'love', 'learning', 'about', 'ai', '[SEP]']
Attention mask: [1, 1, 1, 1, 1, 1, 1]


In [75]:
batch = bert_tokenizer(
    texts,
    padding="max_length",
    max_length=9,
    return_tensors="pt"
)

print("Shape:", batch["input_ids"].shape)

for i in range(len(texts)):
    print(f"\nInput {i + 1}")
    print("Tokens:", bert_tokenizer.convert_ids_to_tokens(
        batch["input_ids"][i]
    ))
    print("Attention mask:", batch["attention_mask"][i].tolist())

Shape: torch.Size([2, 9])

Input 1
Tokens: ['[CLS]', 'i', 'love', 'ai', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
Attention mask: [1, 1, 1, 1, 1, 0, 0, 0, 0]

Input 2
Tokens: ['[CLS]', 'i', 'love', 'learning', 'about', 'ai', '[SEP]', '[PAD]', '[PAD]']
Attention mask: [1, 1, 1, 1, 1, 1, 1, 0, 0]


**Truncation**

In [76]:
texts = [
    "I love AI",
    "I love learning about AI"
]

batch = bert_tokenizer(
    texts,
    padding="max_length",
    truncation=True,
    max_length=5,
    return_tensors="pt"
)

print("Shape:", batch["input_ids"].shape)

for i in range(len(texts)):
    print(f"\nInput {i + 1}")

    print("Tokens:", bert_tokenizer.convert_ids_to_tokens(
        batch["input_ids"][i]
    ))

    print("Attention mask:", batch["attention_mask"][i].tolist())

Shape: torch.Size([2, 5])

Input 1
Tokens: ['[CLS]', 'i', 'love', 'ai', '[SEP]']
Attention mask: [1, 1, 1, 1, 1]

Input 2
Tokens: ['[CLS]', 'i', 'love', 'learning', '[SEP]']
Attention mask: [1, 1, 1, 1, 1]


**Chat Templates**

In [77]:
messages = [
    {
        "role": "user",
        "content": "What is BPE?"
    }
]

formatted_prompt = smollm_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print("Formatted prompt:")
print(repr(formatted_prompt))

Formatted prompt:
'<|im_start|>system\nYou are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>\n<|im_start|>user\nWhat is BPE?<|im_end|>\n<|im_start|>assistant\n'


In [78]:
messages = [
    {"role": "system", "content": "You are an AI tutor."},
    {"role": "user", "content": "Explain BPE."}
]

formatted_prompt = smollm_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print(repr(formatted_prompt))

'<|im_start|>system\nYou are an AI tutor.<|im_end|>\n<|im_start|>user\nExplain BPE.<|im_end|>\n<|im_start|>assistant\n'


avoiding duplicate special tokens when using chat templates.


There are two correct approaches.


**Approach A — Let the chat template tokenize directly**

In [79]:
ids_a = smollm_tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True
)

**Approach B — Format first, then tokenize separately**

In [80]:
formatted_prompt = smollm_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

ids_b = smollm_tokenizer.encode(
    formatted_prompt,
    add_special_tokens=False
)

In [82]:
from collections.abc import Mapping

a_ids = ids_a["input_ids"] if isinstance(ids_a, Mapping) else ids_a
b_ids = ids_b["input_ids"] if isinstance(ids_b, Mapping) else ids_b

print("Approach A token count:", len(a_ids))
print("Approach B token count:", len(b_ids))

print("Token IDs identical:", a_ids == b_ids)

print(
    "Tokens:",
    smollm_tokenizer.convert_ids_to_tokens(a_ids)
)

Approach A token count: 24
Approach B token count: 24
Token IDs identical: True
Tokens: ['<|im_start|>', 'system', 'Ċ', 'You', 'Ġare', 'Ġan', 'ĠAI', 'Ġtutor', '.', '<|im_end|>', 'Ċ', '<|im_start|>', 'user', 'Ċ', 'Explain', 'ĠB', 'PE', '.', '<|im_end|>', 'Ċ', '<|im_start|>', 'ass', 'istant', 'Ċ']
